### MICrONS Neural Data Analysis (NDA)
> Embedding Neuronal Constraints Drive Superior Learning in Recurrent Neural Networks

In [ ]:
spikes_path = 'spikes_6_6_2.npy'

##### SETUP

**Import Libraries**

In [ ]:
# ! pip install numpy==1.26.4
# ! pip install elephant neo quantities --quiet

In [ ]:
from elephant.spike_train_correlation import correlation_coefficient, spike_time_tiling_coefficient #type: ignore
from elephant.conversion import BinnedSpikeTrain #type: ignore
import neo #type: ignore
import numpy as np #type: ignore
import quantities as pq #type: ignore
from tqdm import tqdm  #type: ignore
from scipy.sparse import lil_matrix  #type: ignore
from sklearn.covariance import GraphicalLassoCV #type: ignore
# from google.colab import files

**Load SPIKES data**

In [ ]:
SPIKES = np.load(spikes_path, allow_pickle=True)

In [ ]:
SPIKES.shape

**Create Spike Trains**

In [ ]:
end_time = 0
for sp in SPIKES:
    if sp[-1] > end_time:
        end_time = sp[-1]
print(f"The end time of the last spike is {end_time:.2f} s")

In [ ]:
t_stop = end_time + 0.1

spiketrains = [
    neo.SpikeTrain(spikes, t_stop=t_stop, units='s') for spikes in SPIKES
]
# Now bin the spike trains
bin_size = 20 * pq.ms
bst = BinnedSpikeTrain(spiketrains, bin_size=bin_size)

##### `1.Correlation`

In [ ]:
corr_matrix = correlation_coefficient(bst)

##### `2.Spike Time Tiling`

In [ ]:
# Create a sparse matrix to store the spike time tiling coefficients
num_neurons = len(spiketrains)
sttc_matrix = lil_matrix((num_neurons, num_neurons))

# Calculate the spike time tiling coefficient for the lower triangle
for i in tqdm(range(num_neurons)):
    for j in range(i+1):
        sttc_matrix[i, j] = spike_time_tiling_coefficient(spiketrains[i], spiketrains[j])

##### `3.Precision (Inv. Covariance)`

In [ ]:
estimator = GraphicalLassoCV()
estimator.fit(bst.to_array().T)

##### SAVE

In [ ]:
# save the correlation matrix and the spike time tiling coefficient matrix
np.save('corr_matrix.npy', corr_matrix)
np.save('sttc_matrix.npy', sttc_matrix.toarray())
np.save('precision_matrix.npy', estimator.precision_)

In [ ]:
# files.download('corr_matrix.npy')
# files.download('sttc_matrix.npy')
# files.download('precision_matrix.npy')

---